# 08: ASSUME Simulation Results Analysis

## Learning Goals | 学习目标

1. Read ASSUME simulation output files (CSV)
2. Visualize clearing prices, generation dispatch, and agent profits
3. Compare scenarios (baseline / high wind / summer peak)
4. Understand market clearing mechanism and supply-demand balance
5. Analyze the impact of renewable penetration on electricity prices

## Data Source | 数据来源

This notebook uses output files from `assume/run_simulation.py`.
Each simulation run produces 4 files in the output directory:
- `clearing_prices.csv` -- hourly clearing prices
- `dispatch.csv` -- generation dispatch per unit
- `agent_profits.csv` -- cumulative profit per agent
- `simulation_metadata.json` -- config summary, runtime, status

本 notebook 分析 ASSUME 仿真输出结果。运行仿真后生成 4 个文件。

### Running Simulations First | 先运行仿真

```bash
cd ellectric
python assume/run_simulation.py --config assume/configs/assume_china_config.yaml --output outputs/base
python assume/run_simulation.py --config assume/configs/assume_china_wind_high.yaml --output outputs/wind_high
python assume/run_simulation.py --config assume/configs/assume_china_summer_peak.yaml --output outputs/summer_peak
```

If simulations haven't been run, this notebook will create synthetic demo data
so you can still learn the visualization patterns.

In [ ]:
# === Imports | 导入 ===
import pandas as pd
import numpy as np
import json
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

---
## Step 1: Load Simulation Results | 加载仿真结果

Define a helper to load all 4 output files from a scenario directory.
If the directory doesn't exist, generate synthetic demo data for learning.

定义辅助函数加载仿真输出。如果目录不存在，生成演示数据用于学习。

In [ ]:
def load_simulation_results(scenario_dir):
    """Load all output files from an ASSUME simulation run.
    加载 ASSUME 仿真运行的所有输出文件。

    Returns:
        dict with keys: prices, dispatch, profits, metadata
    """
    d = Path(scenario_dir)
    if not d.exists():
        raise FileNotFoundError(f"Scenario directory not found: {scenario_dir}")

    prices = pd.read_csv(d / 'clearing_prices.csv', parse_dates=['timestamp'])
    dispatch = pd.read_csv(d / 'dispatch.csv', parse_dates=['timestamp'])
    profits = pd.read_csv(d / 'agent_profits.csv', parse_dates=['timestamp'])

    with open(d / 'simulation_metadata.json') as f:
        metadata = json.load(f)

    return {
        'prices': prices, 'dispatch': dispatch,
        'profits': profits, 'metadata': metadata
    }


def create_demo_scenario(name, n_hours=168):
    """Create synthetic demo data when real simulation output is unavailable.
    当真实仿真数据不可用时，创建演示数据。"""
    rng = np.random.default_rng(hash(name) % 2**31)
    base = pd.date_range('2024-07-01', periods=n_hours, freq='h')

    # Price profile: base load ~300, peaks at morning/evening
    hour = np.arange(n_hours) % 24
    price_base = 300 + 150 * np.sin(np.pi * (hour - 6) / 12)
    price_base = np.clip(price_base + rng.normal(0, 50, n_hours), 0, 1500)

    if name == 'wind_high':
        price_base *= 0.7  # high wind -> lower prices
    elif name == 'summer_peak':
        price_base *= 1.4  # peak demand -> higher prices

    prices = pd.DataFrame({
        'timestamp': base,
        'price_da_元_per_mwh': price_base,
    })

    # Dispatch: coal baseload, wind/solar variable, gas peaking
    coal = 30000 + rng.normal(0, 1000, n_hours)
    wind = 8000 + 4000 * np.sin(np.pi * (hour - 3) / 12) + rng.normal(0, 2000, n_hours)
    wind = np.clip(wind, 0, 15000)
    solar = np.clip(6000 * np.sin(np.pi * (hour - 6) / 12), 0, 8000) + rng.normal(0, 500, n_hours)
    solar = np.clip(solar, 0, 10000)
    gas = np.clip(5000 + 3000 * np.sin(np.pi * (hour - 10) / 12) + rng.normal(0, 1000, n_hours), 0, 10000)

    if name == 'wind_high':
        wind *= 1.8
        coal *= 0.6
    elif name == 'summer_peak':
        coal *= 1.2
        gas *= 1.5

    dispatch_rows = []
    for i, t in enumerate(base):
        for fuel, val in [('coal', coal[i]), ('gas', gas[i]),
                           ('wind', wind[i]), ('solar', solar[i])]:
            dispatch_rows.append({'timestamp': t, 'fuel_type': fuel, 'dispatch_mw': val})
    dispatch = pd.DataFrame(dispatch_rows)

    # Agent profits
    profits_rows = []
    for i, t in enumerate(base):
        for agent, mult in [('learning', 1.5), ('naive', 1.0), ('strategic', 2.0)]:
            profit = (price_base[i] - 250) * 100 * mult / 1000 + rng.normal(0, 50)
            profits_rows.append({'timestamp': t, 'agent': agent, 'profit_元': profit})
    profits = pd.DataFrame(profits_rows)

    metadata = {
        'scenario': name, 'hours': n_hours,
        'note': 'DEMO DATA -- run run_simulation.py for real results',
        'note_cn': '演示数据 -- 运行 run_simulation.py 获取真实结果'
    }

    return {'prices': prices, 'dispatch': dispatch, 'profits': profits, 'metadata': metadata}


# --- Load scenarios (real or demo) ---
scenarios = {}
is_demo = False
output_base = Path('../outputs')

for name in ['base', 'wind_high', 'summer_peak']:
    path = output_base / name
    try:
        scenarios[name] = load_simulation_results(str(path))
        print(f"[OK] {name}: loaded ({len(scenarios[name]['prices'])} hours)")
    except FileNotFoundError:
        is_demo = True
        scenarios[name] = create_demo_scenario(name)
        print(f"[DEMO] {name}: synthetic data ({len(scenarios[name]['prices'])} hours) -- run simulation for real data")

if is_demo:
    print(f"\nNote: Using DEMO data. Run 'python assume/run_simulation.py' for real results.")
    print("注意: 使用演示数据。运行仿真脚本获取真实结果。")

---
## Step 2: Clearing Price Analysis | 出清价格分析

Plot clearing price time series across scenarios.

### Key Concepts | 关键概念
- **Clearing Price** = marginal cost of the last accepted unit (pay-as-clear)
- **Price Cap** = 1500 RMB/MWh (China provincial spot market rule)
- **Price Floor** = 0 RMB/MWh
- **Summer Peak**: higher demand -> more expensive units (gas) dispatched -> price rises
- **High Wind**: abundant zero-marginal-cost wind -> pushes expensive units out -> price drops

出清价格 = 最后一个中标机组的边际成本。价格上限 1500 元/MWh (中国省间现货规则)。
夏季高峰推高价格，高风电压低价格。

In [ ]:
# --- Price Time Series Comparison ---
scenario_colors = {'base': '#1f77b4', 'wind_high': '#2ca02c', 'summer_peak': '#d62728'}

fig1 = go.Figure()

for name, data in scenarios.items():
    df = data['prices']
    price_col = 'price_da_元_per_mwh'
    if price_col not in df.columns:
        # fallback: try other common price column names
        for alt in ['price_da', 'clearing_price', 'price']:
            if alt in df.columns:
                price_col = alt
                break

    fig1.add_trace(go.Scatter(
        x=df['timestamp'], y=df[price_col],
        mode='lines', name=f"{name} ({data['metadata'].get('scenario', name)})",
        line=dict(width=2, color=scenario_colors.get(name, '#999')),
    ))

fig1.add_hline(y=1500, line_dash='dash', line_color='red',
               annotation_text='Price Cap (1500 RMB/MWh)')
fig1.add_hline(y=0, line_dash='dash', line_color='gray',
               annotation_text='Price Floor (0 RMB/MWh)')

fig1.update_layout(
    title='Clearing Prices by Scenario -- 7-Day ASSUME Simulation',
    xaxis_title='Time', yaxis_title='Price (RMB/MWh)',
    hovermode='x unified', height=500,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig1.show()

### Price Statistics | 价格统计

In [ ]:
# --- Price Statistics Table ---
print(f"{'Scenario':<15} {'Mean':>10} {'Min':>10} {'Max':>10} {'Std':>10} {'Hours>=1000':>12}")
print('-' * 70)
for name, data in scenarios.items():
    df = data['prices']
    price_col = 'price_da_元_per_mwh'
    if price_col not in df.columns:
        for alt in ['price_da', 'clearing_price', 'price']:
            if alt in df.columns:
                price_col = alt
                break
    p = df[price_col]
    print(f"{name:<15} {p.mean():>10.0f} {p.min():>10.0f} {p.max():>10.0f} {p.std():>10.0f} {(p >= 1000).sum():>12}")

# Price distribution comparison
fig_price_dist = go.Figure()
for name, data in scenarios.items():
    df = data['prices']
    price_col = 'price_da_元_per_mwh'
    if price_col not in df.columns:
        for alt in ['price_da', 'clearing_price', 'price']:
            if alt in df.columns:
                price_col = alt
                break
    p = df[price_col]
    fig_price_dist.add_trace(go.Histogram(
        x=p, nbinsx=30, name=name,
        marker_color=scenario_colors.get(name, '#999'),
        opacity=0.6,
    ))

fig_price_dist.update_layout(
    title='Price Distribution by Scenario',
    xaxis_title='Price (RMB/MWh)', yaxis_title='Hours',
    barmode='overlay', height=400,
)
fig_price_dist.show()

---
## Step 3: Generation Dispatch Analysis | 发电调度分析

Generator dispatch follows **Merit Order**:
1. Renewables (wind/solar) dispatched first -- marginal cost near zero
2. Coal as baseload (~300 RMB/MWh)
3. Gas only during peak hours (~600 RMB/MWh)

发电机按边际成本排序依次调用。新能源优先（边际成本接近零），
煤电基荷，气电仅在高峰调用。

In [ ]:
# --- Dispatch Stacked Area Chart ---
target = 'base'
if target not in scenarios:
    target = list(scenarios.keys())[0]

dispatch = scenarios[target]['dispatch']

# Check available columns
if 'fuel_type' in dispatch.columns and 'dispatch_mw' in dispatch.columns:
    pivot = dispatch.pivot_table(
        index='timestamp', columns='fuel_type', values='dispatch_mw', aggfunc='sum'
    ).fillna(0)

    fuel_colors_map = {
        'coal': '#8B4513', 'gas': '#FF8C00',
        'wind': '#2E8B57', 'solar': '#FFD700',
        'storage': '#9370DB', 'nuclear': '#4682B4',
    }

    fig2 = go.Figure()
    for col in pivot.columns:
        fig2.add_trace(go.Scatter(
            x=pivot.index, y=pivot[col],
            mode='lines', name=col,
            stackgroup='one',
            line=dict(width=0.5, color=fuel_colors_map.get(col, '#999')),
        ))

    fig2.update_layout(
        title=f'Generation Dispatch by Fuel Type -- {target} Scenario',
        xaxis_title='Time', yaxis_title='Dispatch (MW)',
        hovermode='x unified', height=450,
    )
    fig2.show()
else:
    print(f"Dispatch columns: {list(dispatch.columns)}")
    print("Cannot create stacked chart -- unexpected column names")
    print("无法创建堆叠图 -- 列名不匹配")

### Renewable Share Comparison | 新能源消纳率对比

In [ ]:
# --- Renewable Share by Scenario ---
print(f"{'Scenario':<15} {'Renewable Share':>18}")
print('-' * 35)

renewable_data = []
for name, data in scenarios.items():
    d = data['dispatch']
    if 'fuel_type' in d.columns and 'dispatch_mw' in d.columns:
        total = d['dispatch_mw'].sum()
        re = d[d['fuel_type'].isin(['wind', 'solar'])]['dispatch_mw'].sum()
        share = re / total * 100 if total > 0 else 0
        renewable_data.append({'Scenario': name, 'Share': share})
        print(f"{name:<15} {share:>17.1f}%")
    else:
        print(f"{name:<15} {'N/A':>18}")

if renewable_data:
    fig_re = go.Figure()
    fig_re.add_trace(go.Bar(
        x=[r['Scenario'] for r in renewable_data],
        y=[r['Share'] for r in renewable_data],
        marker_color=[scenario_colors.get(r['Scenario'], '#999') for r in renewable_data],
        text=[f"{r['Share']:.1f}%" for r in renewable_data],
        textposition='outside',
    ))
    fig_re.update_layout(
        title='Renewable Share by Scenario',
        yaxis_title='Renewable Share (%)',
        height=350, showlegend=False,
    )
    fig_re.show()

---
## Step 4: Agent Profit Analysis | 智能体利润分析

Three agent types in ASSUME:
- **learning** (PPO): reinforcement learning agent, dynamic bidding
- **naive**: bids at marginal cost
- **strategic**: strategic mark-up bidding

三种智能体类型：learning (PPO 强化学习)、naive (按边际成本报价)、strategic (策略性加价)。

In [ ]:
# --- Total Profit Comparison ---
fig3 = go.Figure()

for name, data in scenarios.items():
    profits = data['profits']
    if 'agent' in profits.columns and 'profit_元' in profits.columns:
        agent_totals = profits.groupby('agent')['profit_元'].sum().reset_index()
        fig3.add_trace(go.Bar(
            name=name,
            x=agent_totals['agent'],
            y=agent_totals['profit_元'],
            marker_color=scenario_colors.get(name, '#999'),
            text=agent_totals['profit_元'].apply(lambda x: f'{x:,.0f}'),
            textposition='outside',
        ))

fig3.update_layout(
    title='Agent Total Profit Comparison by Scenario',
    xaxis_title='Agent', yaxis_title='Total Profit (RMB)',
    barmode='group', height=450,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig3.show()

### Cumulative Profit Over Time | 累计利润时序

How does each agent's profit evolve during the simulation?
各智能体利润如何随时间变化？

In [ ]:
# --- Cumulative Profit by Agent ---
if target in scenarios:
    profits = scenarios[target]['profits']

    if 'agent' in profits.columns and 'profit_元' in profits.columns:
        fig4 = go.Figure()

        agent_colors = {'learning': '#1f77b4', 'naive': '#ff7f0e', 'strategic': '#2ca02c'}
        for agent in profits['agent'].unique():
            agent_data = profits[profits['agent'] == agent].sort_values('timestamp')
            cumulative = agent_data['profit_元'].cumsum()
            fig4.add_trace(go.Scatter(
                x=agent_data['timestamp'], y=cumulative,
                mode='lines', name=agent,
                line=dict(width=2, color=agent_colors.get(agent, '#999')),
            ))

        fig4.update_layout(
            title=f'Cumulative Profit by Agent -- {target} Scenario',
            xaxis_title='Time', yaxis_title='Cumulative Profit (RMB)',
            hovermode='x unified', height=450,
        )
        fig4.show()
    else:
        print(f"Profit columns: {list(profits.columns)}")
        print("Cannot create profit chart -- unexpected column names")
else:
    print(f"Scenario '{target}' not found")

---
## Step 5: Scenario Comparison Summary | 场景对比摘要

Compare key metrics across all three scenarios.

In [ ]:
# --- Cross-Scenario Summary ---
print("=" * 70)
print("Cross-Scenario Comparison | 场景对比摘要")
print("=" * 70)
print()

for name, data in scenarios.items():
    prices = data['prices']
    price_col = 'price_da_元_per_mwh'
    if price_col not in prices.columns:
        for alt in ['price_da', 'clearing_price', 'price']:
            if alt in prices.columns:
                price_col = alt
                break

    p = prices[price_col]
    meta = data['metadata']

    print(f"Scenario: {name}")
    print(f"  Hours:         {len(prices)}")
    print(f"  Avg Price:     {p.mean():.0f} RMB/MWh")
    print(f"  Price Range:   [{p.min():.0f}, {p.max():.0f}]")
    print(f"  Price Vol:     {p.std():.0f} (std)")
    print(f"  Hours >= 1000: {(p >= 1000).sum()} ({((p >= 1000).sum()/len(p)*100):.1f}%)")

    # Renewable share
    d = data['dispatch']
    if 'fuel_type' in d.columns and 'dispatch_mw' in d.columns:
        total = d['dispatch_mw'].sum()
        re = d[d['fuel_type'].isin(['wind', 'solar'])]['dispatch_mw'].sum()
        print(f"  Renewable %:   {re/total*100:.1f}%" if total > 0 else "")

    # Top agent
    profits = data['profits']
    if 'agent' in profits.columns and 'profit_元' in profits.columns:
        agent_totals = profits.groupby('agent')['profit_元'].sum()
        top_agent = agent_totals.idxmax()
        print(f"  Top Agent:     {top_agent} ({agent_totals[top_agent]:,.0f} RMB)")

    if 'note' in meta or 'note_cn' in meta:
        print(f"  Note:          {meta.get('note', meta.get('note_cn', ''))}")
    print()

---
## Step 6: Price vs Renewable Correlation | 价格与新能源相关性

High renewable output should correlate with lower prices
(merit-order effect). Let's check this relationship.

高新能源出力应该与低电价相关（merit-order 效应）。检查这一关系。

In [ ]:
# --- Price vs Renewable Scatter ---
if target in scenarios:
    prices = scenarios[target]['prices']
    dispatch = scenarios[target]['dispatch']

    price_col = 'price_da_元_per_mwh'
    if price_col not in prices.columns:
        for alt in ['price_da', 'clearing_price', 'price']:
            if alt in prices.columns:
                price_col = alt
                break

    if 'fuel_type' in dispatch.columns:
        # Aggregate renewable generation per timestamp
        re_total = dispatch[dispatch['fuel_type'].isin(['wind', 'solar'])].groupby('timestamp')['dispatch_mw'].sum()
        total_gen = dispatch.groupby('timestamp')['dispatch_mw'].sum()
        re_share = (re_total / total_gen * 100).fillna(0)

        # Merge with prices
        merged = pd.merge(
            prices[['timestamp', price_col]],
            re_share.rename('re_share_pct').reset_index(),
            on='timestamp', how='inner'
        )

        fig_corr = go.Figure()
        fig_corr.add_trace(go.Scatter(
            x=merged['re_share_pct'], y=merged[price_col],
            mode='markers',
            marker=dict(color='#1f77b4', size=6, opacity=0.5),
            name='Hourly observation',
        ))

        # Add trend line
        if len(merged) > 5:
            z = np.polyfit(merged['re_share_pct'], merged[price_col], 1)
            p_trend = np.poly1d(z)
            x_range = np.linspace(merged['re_share_pct'].min(), merged['re_share_pct'].max(), 100)
            fig_corr.add_trace(go.Scatter(
                x=x_range, y=p_trend(x_range),
                mode='lines', name=f'Trend (slope={z[0]:.1f})',
                line=dict(color='#d62728', width=2, dash='dash'),
            ))
            corr = merged['re_share_pct'].corr(merged[price_col])
            print(f"Price vs Renewable Share correlation: {corr:.3f}")
            print(f"Slope: {z[0]:.1f} RMB/MWh per % renewable")
            if corr < 0:
                print("Negative correlation confirms merit-order effect!")
                print("负相关证实了 merit-order 效应！新能源越多 -> 电价越低")

        fig_corr.update_layout(
            title=f'Price vs Renewable Share -- {target} Scenario',
            xaxis_title='Renewable Share (%)',
            yaxis_title='Clearing Price (RMB/MWh)',
            height=400,
        )
        fig_corr.show()
    else:
        print("Cannot create scatter -- unexpected dispatch format")

---
## Key Takeaways | 学习笔记

- **Merit Order**: generators sorted by marginal cost, lowest wins first
  Merit Order: 发电机组按边际成本排序，最低者优先中标
- **Pay-as-Clear**: all accepted units receive the marginal unit's price
  所有中标机组按边际机组的报价结算（统一价格）
- **Price Cap**: China provincial spot price cap = 1500 RMB/MWh
  中国省间现货报价上限 1500 元/MWh
- **Renewables First**: wind/solar marginal cost near zero, always at top of merit order
  新能源优先: 风光边际成本接近零，在 merit order 中永远排最前
- **Scenario Value**: high wind scenarios lower average prices; summer peaks push them up
  场景对比价值: 高风电场景降低平均出清价格; 夏季高峰推高价格

## Reflection Questions | 反思题

1. Why does the wind_high scenario have lower average prices than base?
   为什么 wind_high 场景的平均价格低于 base？
2. Which generators are added during summer peaks? What are their marginal costs?
   夏季高峰时哪些机组新增了发电量？边际成本有什么特点？
3. Why does the strategic agent out-earn the naive one?
   为什么 strategic 智能体的利润高于 naive？
4. What happens to prices if the 1500 RMB/MWh cap is removed under extreme scarcity?
   如果去掉价格上限，极端供需紧张时价格会怎样变化？
5. How could the Shandong 15-min data from notebooks 05-07 be fed into ASSUME?
   如何将 notebook 05-07 的山东 15min 数据导入 ASSUME 仿真？

---
## Next Steps | 下一步

Complete this notebook series:
- Notebook 09: RL Trading Agent (PPO/SAC/TD3)
- Notebook 10: Multi-Agent Backtest
- Notebook 11: Model Explainability (SHAP)

Or return to the learning path and deepen any topic.